# Tools to check and cleant PCAP results

- Size
- Name filters

In [ ]:
from pathlib import Path
import shutil

In [ ]:
# Move PCAP files from PCAP_DIR to PCAP_DIR_CLEAN if they match size and substring criteria.
# Requires the following variables to be defined in the notebook before running this cell:
# PCAP_DIR, PCAP_DIR_CLEAN, PCAP_SIZE_MIN, PCAP_SIZE_MAX, PCAP_SUBSTING

PCAP_DIR = "/Users/tarik/data/pcap"
PCAP_DIR_CLEAN = "/Users/tarik/data/failed/pcap"
PCAP_SIZE_MIN = 30_000        # Minimum size in bytes
PCAP_SIZE_MAX = 10_000_000  # Maximum size in bytes
PCAP_SUBSTING = "e"     # Substring to look for in filenames

DRY_RUN = False  # Set to False to actually move files

In [ ]:
required = ("PCAP_DIR", "PCAP_DIR_CLEAN", "PCAP_SIZE_MIN", "PCAP_SIZE_MAX", "PCAP_SUBSTING")
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"Please define the following variables in the notebook before running this cell: {', '.join(missing)}")

src = Path(PCAP_DIR)
dst = Path(PCAP_DIR_CLEAN)
dst.mkdir(parents=True, exist_ok=True)

moved = []
for p in src.iterdir():
    if not p.is_file():
        continue
    # consider common pcap extensions
    if p.suffix.lower() not in (".pcap", ".pcapng"):
        continue
    if PCAP_SUBSTING not in p.name:
        continue
    try:
        size = p.stat().st_size
    except OSError:
        continue
    if not PCAP_SIZE_MIN <= size <= PCAP_SIZE_MAX:
        dest = dst / p.name
        if not DRY_RUN:
            shutil.move(str(p), str(dest))
        moved.append((p.name, size))

print(f"Moved {len(moved)} files from {src} to {dst}")
for name, size in moved:
    print(f"- {name} ({size} bytes)")